In [2]:
pip install garminconnect garth pandas


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import datetime
import pandas as pd
from garminconnect import Garmin
import garth
from getpass import getpass

# --- CONFIGURATION ---
EMAIL = "shaunrussell1111@gmail.com"
# For security, you'll be prompted for your password in the console
PASSWORD = getpass("Enter your Garmin Password: ")
DAYS_TO_COLLECT = 30  # Adjust for your FYP timeframe
TOKEN_STORE = "~/.garth" # Saves login so you don't need MFA every time

def get_client():
    client = Garmin(EMAIL, PASSWORD)
    try:
        # Try to resume an existing session to avoid repeated logins
        client.garth.resume(TOKEN_STORE)
    except Exception:
        # If no session exists, login (will prompt for MFA code if enabled)
        client.login()
        client.garth.dump(TOKEN_STORE)
    return client

def collect_burnout_data(days):
    client = get_client()
    end_date = datetime.date.today()
    start_date = end_date - datetime.timedelta(days=days)
    
    data_list = []
    current_date = start_date

    print(f"Starting data collection for {days} days...")
    
    while current_date <= end_date:
        date_str = current_date.isoformat()
        print(f"Fetching: {date_str}")
        
        try:
            # 1. Fetch Stress & Body Battery
            # These are core metrics for burnout detection
            stats = client.get_stats(date_str)
            
            # 2. Fetch HRV Data (Heart Rate Variability)
            # Critical for Autonomic Nervous System (ANS) analysis
            hrv_data = client.get_hrv_data(date_str)
            hrv_value = hrv_data.get('hrvSummary', {}).get('lastNightAvg', None) if hrv_data else None
            
            # 3. Fetch Sleep Score
            sleep_data = client.get_sleep_data(date_str)
            sleep_score = sleep_data.get('dailySleepDTO', {}).get('sleepScore', None) if sleep_data else None

            # Append only essential features for your AI model
            data_list.append({
                "date": date_str,
                "avg_stress": stats.get('averageStressLevel'),
                "max_stress": stats.get('maxStressLevel'),
                "rest_hr": stats.get('restingHeartRate'),
                "hrv_avg": hrv_value,
                "sleep_score": sleep_score,
                "body_battery_max": stats.get('bodyBatteryHighestValue')
            })
        except Exception as e:
            print(f"Error on {date_str}: {e}")
            
        current_date += datetime.timedelta(days=1)

    # Convert to DataFrame and save
    df = pd.DataFrame(data_list)
    df.to_csv("garmin_burnout_data.csv", index=False)
    print("Success! Data saved to 'garmin_burnout_data.csv'")

if __name__ == "__main__":
    collect_burnout_data(DAYS_TO_COLLECT)


Enter your Garmin Password:  ········


Starting data collection for 30 days...
Fetching: 2026-01-20
Fetching: 2026-01-21
Fetching: 2026-01-22
Fetching: 2026-01-23
Fetching: 2026-01-24
Fetching: 2026-01-25
Fetching: 2026-01-26
Fetching: 2026-01-27
Fetching: 2026-01-28
Fetching: 2026-01-29
Fetching: 2026-01-30
Fetching: 2026-01-31
Fetching: 2026-02-01
Fetching: 2026-02-02
Fetching: 2026-02-03
Fetching: 2026-02-04
Fetching: 2026-02-05
Fetching: 2026-02-06
Fetching: 2026-02-07
Fetching: 2026-02-08
Fetching: 2026-02-09
Fetching: 2026-02-10
Fetching: 2026-02-11
Fetching: 2026-02-12
Fetching: 2026-02-13
Fetching: 2026-02-14
Fetching: 2026-02-15
Fetching: 2026-02-16
Fetching: 2026-02-17
Fetching: 2026-02-18
Fetching: 2026-02-19
Success! Data saved to 'garmin_burnout_data.csv'
